# From a point cloud to comparisons and features

This tutorial is for a first ordinary-persistence analysis. You only need Julia, TamerOp, and the idea that degree zero records connected components. We will compute two diagrams, compare them, make numerical features, and save a result with its retained cocycle data.

Use the Julia kernel with this repository as its active project (`julia --project=.` before starting your notebook environment). No plotting package is required. The cells use the public API; the assertions check the answers we can derive by hand.

1. Predict the component merges.
2. Compute and compare the diagrams.
3. Choose what to do with the component that survives forever.
4. Make landscapes, images and summary statistics on common grids.
5. Save and recover the result.


In [1]:
using TamerOp
using TamerOp.Advanced: landscape_grid, landscape_values, image_values,
    persistence_cocycle, field, filtration_order, check_persistence_diagram_json


## Three points and two merges

Put three points at 0, 2 and 5 on a line. At radius zero they are three separate components. At radius 2 the first two merge, and at radius 3 the third joins them. Thus we expect two finite intervals, `[0,2)` and `[0,3)`, and one essential interval `[0,infinity)`.

Moving the last point to 6 changes the second merge to radius 4. We keep the same filtration and homology degree for both clouds.


In [2]:
cloud = PointCloud([[0.0], [2.0], [5.0]])
changed_cloud = PointCloud([[0.0], [2.0], [6.0]])
filtration = RipsFiltration(max_dim=1)


RipsFiltration
  kind: rips
  arity: 1
  key_params: (max_dim = 1, nn_backend = :auto, nn_approx_candidates = 0)
  construction_mode: (sparsify = :none, collapse = :none, output_stage = :encoding_result)

Request cocycles so the saved result later includes an example of retained source information. This costs extra work and is optional for all the interval-based analysis below.

In [3]:
diagram = persistence_diagram(cloud, filtration; max_homology_dim=0, cocycles=true)
changed = persistence_diagram(changed_cloud, filtration; max_homology_dim=0)
finite_intervals(diagram; dim=0)


2-element Vector{Tuple{Float64, Float64}}:
 (0.0, 2.0)
 (0.0, 3.0)

In [4]:
@assert sort(finite_intervals(diagram; dim=0)) == [(0.0, 2.0), (0.0, 3.0)]
@assert sort(finite_intervals(changed; dim=0)) == [(0.0, 2.0), (0.0, 4.0)]
@assert essential_births(diagram; dim=0) == [0.0]
summary = describe(diagram)
(finite=summary.finite_counts, essential=summary.essential_counts,
    cocycles_retained=summary.cocycles_available)


(finite = (2,), essential = (1,), cocycles_retained = true)

## Compare the same degree

A bottleneck comparison minimizes the largest interval-matching cost; Wasserstein combines all matching costs. Here one death changes by 1, so both distances are 1. The unchanged essential component matches at cost zero. Distances keep essential intervals by default; unequal essential counts would give infinity.

A barcode matching compares intervals. It does not identify chains or provide a map between the two point clouds.


In [5]:
bottleneck = bottleneck_distance(diagram, changed; dim=0)
wasserstein = wasserstein_distance(diagram, changed; dim=0, p=2)
@assert bottleneck == wasserstein == 1.0
(bottleneck=bottleneck, wasserstein=wasserstein)


(bottleneck = 1.0, wasserstein = 1.0)

## Make the essential-bar choice explicit

Finite features need finite lifetimes. We choose `essential=:drop` to describe the component mergers. That omits the one component that survives forever. The diagram itself remains intact. `analysis_barcode` lets us inspect the exact increasing-time intervals passed to the shared algorithms.

Alternatively, `essential=:cap, essential_cap=6` gives that component an artificial death at 6. This is a chosen observation convention, not a computed death or extended persistence.


In [6]:
analysis_barcode(diagram; dim=0, essential=:drop)


2-element Vector{Tuple{Rational{BigInt}, Rational{BigInt}}}:
 (0, 2)
 (0, 3)

In [7]:
capped = analysis_barcode(diagram; dim=0, essential=:cap, essential_cap=6)
@assert sort(capped) == [(0, 2), (0, 3), (0, 6)]
capped


3-element Vector{Tuple{Rational{BigInt}, Rational{BigInt}}}:
 (0, 2)
 (0, 3)
 (0, 6)

## Landscapes on a common grid

Each finite interval contributes a tent: zero outside the interval and height `min(t-birth, death-t)` inside. The first landscape layer takes the highest tent at each time, the next layer the second highest. Use the same grid for both clouds to make their feature arrays comparable. At time 1 our two tents both have height 1.


In [8]:
grid = 0.0:0.5:4.0
landscape = persistence_landscape(diagram; dim=0, essential=:drop, tgrid=grid, kmax=2)
changed_landscape = persistence_landscape(changed; dim=0, essential=:drop, tgrid=grid, kmax=2)
@assert landscape_values(landscape)[:, 3] == [1.0, 1.0]
landscape_values(landscape)


2×9 Matrix{Float64}:
 0.0  0.5  1.0  1.5  1.0  0.5  0.0  0.0  0.0
 0.0  0.5  1.0  0.5  0.0  0.0  0.0  0.0  0.0

A silhouette averages the tents with chosen weights instead of sorting them into layers. With lifetime weights 2 and 3, the value at time 1 is again 1. At time 2 only the longer interval contributes, giving `3/5`.

In [9]:
silhouette = persistence_silhouette(diagram; dim=0, essential=:drop, tgrid=grid)
@assert silhouette[3] == 1.0 && silhouette[5] == 0.6
silhouette


9-element Vector{Float64}:
 0.0
 0.5
 1.0
 1.1
 0.6
 0.3
 0.0
 0.0
 0.0

## A persistence image and two scalar summaries

An image puts a Gaussian at each (birth, lifetime) point and samples it at grid centers. These are numerical values, not exact pixel integrals. Here both births are zero and the lifetimes are 2 and 3; by default they also supply the weights.

We show the small matrix directly. Its rows index lifetime and columns index birth. Keep coordinates, grid, bandwidth, weighting and normalization fixed when comparing images.


In [10]:
image = persistence_image(diagram; dim=0, essential=:drop,
    xgrid=[0.0, 1.0, 2.0], ygrid=[0.0, 1.0, 2.0, 3.0, 4.0], sigma=0.5)
image_values(image)


5×3 Matrix{Float64}:
 0.000670971  9.0806e-5  2.25086e-7
 0.271677     0.0367675  9.11375e-5
 2.40601      0.325617   0.000807125
 3.27067      0.442637   0.00109719
 0.406677     0.0550377  0.000136425

Total persistence is `2+3=5`. Persistent entropy measures how evenly that lifetime is distributed. We request unnormalized entropy to obtain `-(2/5)log(2/5)-(3/5)log(3/5)`. The default normalized version divides by the logarithm of the number of intervals.

In [11]:
statistics = barcode_summary(diagram; dim=0, essential=:drop)
entropy = barcode_entropy(diagram; dim=0, essential=:drop, normalize=false)
@assert statistics.n_intervals == 2 && statistics.total_persistence == 5.0
@assert isapprox(entropy, -(2/5)*log(2/5)-(3/5)*log(3/5))
(statistics=statistics, entropy=entropy)


(statistics = (n_intervals = 2, total_persistence = 5.0, max_persistence = 3.0, mean_persistence = 2.5, l2_persistence = 3.605551275463989, entropy = 0.9709505944546688), entropy = 0.6730116670092565)

## Save and recover the actual result

Save the diagram rather than a feature vector when you need its exact grades, coefficient field, order, provenance or retained cocycles later. We use a temporary folder so rerunning this tutorial cannot overwrite your own results. Replace it with your chosen destination in an analysis project.


In [12]:
result_folder = mktempdir()
result_path = joinpath(result_folder, "ordinary-diagram.json")
save_persistence_diagram_json(result_path, diagram)
restored = load_persistence_diagram_json(result_path)
@assert check_persistence_diagram_json(result_path).valid
@assert finite_intervals(restored; dim=0) == finite_intervals(diagram; dim=0)
@assert essential_births(restored; dim=0) == essential_births(diagram; dim=0)
@assert field(restored) == field(diagram)
@assert filtration_order(restored) == filtration_order(diagram)
@assert bottleneck_distance(restored, diagram; dim=0) == 0.0
inspect_json(result_path)


JSONArtifactSummary
  path: "/tmp/jl_rhirAu/ordinary-diagram.json"
  kind: "PersistenceDiagram"
  schema_version: 1
  field: "Fp"
  field_characteristic: 2
  order: "sublevel"
  finite_counts: [2]
  essential_counts: [1]
  representatives_available: false
  cocycles_available: true
  profile_hint: :compact
  size_bytes: 18133


At radius 4 all three original points are connected. Recover the retained degree-zero cocycle on that essential component. The readback uses the original point indices. Loading checks stored structure; it does not reconstruct the source cloud or prove its chain equations anew.

In [13]:
original_cocycle = persistence_cocycle(diagram; dim=0, kind=:essential, scale=4)
restored_cocycle = persistence_cocycle(restored; dim=0, kind=:essential, scale=4)
@assert restored_cocycle == original_cocycle
restored_cocycle.source_vertices


((1,), (2,), (3,))

## Try one change

What should the total persistence become if we include the essential component with a cap at 6? Predict it before running the answer cell: the two real lifetimes remain 2 and 3, and the capped lifetime is 6.


In [14]:
capped_statistics = barcode_summary(diagram; dim=0, essential=:cap, essential_cap=6)
@assert capped_statistics.total_persistence == 11.0
capped_statistics


(n_intervals = 3, total_persistence = 11.0, max_persistence = 6.0, mean_persistence = 3.6666666666666665, l2_persistence = 7.0, entropy = 0.9056185178648963)

## Coordinates and next steps

Superlevel diagrams run toward smaller grades. Analysis reflects their time coordinate, using `(origin-grade)/scale`; sublevels use `(grade-origin)/scale`. `scale` is positive, and caps are always supplied in original grade units. Numerical grids use the transformed coordinates. Exact intervals stay exact, while distance outputs and sampled features are numerical.

If large grades collapse to one Float64 grid point, choose a common `origin` and `scale` before sampling. Do not independently normalize each result when the intended comparison uses a common physical parameter.

The [ordinary persistence guide](../ordinary_persistence.md) lists these choices. To make a figure, load a supported plotting extension and use `visualize(diagram; kind=:barcode, dim=0)`. The computations above need no rendering backend.

We have followed one result from its predictable mergers to distances, finite features, and a saved result with retained source data. The next cell removes only the temporary folder made by this notebook.


In [15]:
rm(result_folder; recursive=true)
